# 目的地周邊停車場查詢 Demo（Destination-Radius Parking Lookup）

**目標**：輸入一筆查詢（目的地 + 時間），系統回傳該**目的地周圍 X 公尺半徑內**的停車場資訊
（經緯度、容量、可停機率、預估可用車位），並標註在地圖上。

對應簡易示意圖：以目的地 **B** 為圓心、半徑 **X**（例：附近 1400 公尺），
找出落在圓內的停車場 A、C、D、E、F、G 並依可用性上色。

```
          • B (目的地)
      • C        • A
   (  半徑 X 公尺的搜尋圈  )
      • D    • E
          • G    • F
```

本 Notebook 直接重用專案既有引擎：`GridLatLngMapper`、`ConvLSTM` 人流預測、
`create_poi_parking` 停車機率模型。**預設可離線執行**（停車場 POI 讀快取檔）；
若另設 `OPENAI_API_KEY` 則可改用自然語言查詢（見最後附錄）。

## 1. 環境設定與模組載入

In [1]:
import os, sys

# 將工作目錄切到專案根目錄，使 src 內的相對路徑（data/...）能正確讀取
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
ROOT = os.getcwd()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
print("專案根目錄：", ROOT)

import numpy as np
import pandas as pd
import folium
from dotenv import load_dotenv

from src.geo.grid_to_latlng import GridLatLngMapper, haversine_m_vec, cell_size_m_at
from src.preprocess import add_lags_and_rollings
from src.ConvLSTM import load_convlstm_embed, make_df_pred_convlstm_embed
from src.parks_routing import create_poi_parking, aggregate_df_prob_same_weekday

load_dotenv()
print("模組載入完成")

專案根目錄： c:\Users\USER\Desktop\2026_japan
模組載入完成


In [2]:
# ---- 全域設定（與 app.py 一致）----
CONVLSTM_PATH = "models/convlstm_sapporo.pkl"
PARQUET_PATH  = "data/processed/sapporo_density.parquet"
START_DATE    = "2019-09-15"
CITY_BOUNDS   = (42.9, 140.7, 43.9, 141.6)   # (south, west, north, east)
T_BUFFER      = 4                              # 預測時段往後多取幾格

# 格網 <-> 經緯度 的錨點（札幌一帶）
ANCHORS = [
    {"x": 24,  "y": 151, "lat": 43.0691833, "lng": 141.3514707},  # 札幌車站
    {"x": 24,  "y": 148, "lat": 43.0794037, "lng": 141.3422559},  # 北海道大學
    {"x": 26,  "y": 153, "lat": 43.0579859, "lng": 141.3540211},
    {"x": 52,  "y": 81,  "lat": 43.1982317, "lng": 140.9940363},
    {"x": 50,  "y": 41,  "lat": 43.1880641, "lng": 140.7945541},
    {"x": 182, "y": 186, "lat": 43.8536095, "lng": 141.5234881},
]

## 2. 載入 ConvLSTM 模型與人流資料

In [3]:
df_true = pd.read_parquet(PARQUET_PATH).copy()
model, cfg = load_convlstm_embed(CONVLSTM_PATH)
seq_len = int(cfg["seq_len"])

# 建立滯後特徵，丟掉缺 lag 的列
df_feat = add_lags_and_rollings(df_true.copy(), seq_len=seq_len, start_date=START_DATE)
need_cols = [f"lag_{k}" for k in range(1, seq_len + 1)]
df_feat = df_feat.dropna(subset=need_cols).copy()

mapper = GridLatLngMapper(ANCHORS)
print(f"人流特徵資料：{df_feat.shape}，seq_len={seq_len}")
df_feat.head(3)

人流特徵資料：(1967668, 16)，seq_len=8


,d,t,x,y,count,date,weekday,is_weekend,lag_1,lag_2,lag_3,lag_4,lag_5,lag_6,lag_7,lag_8
573158,7,6,24,153,13,2019-09-22,6,1,10.0,14.0,16.0,24.0,23.0,28.0,16.0,4.0
573159,7,6,23,153,10,2019-09-22,6,1,13.0,10.0,16.0,21.0,16.0,17.0,8.0,7.0
573160,7,6,24,154,8,2019-09-22,6,1,10.0,9.0,15.0,11.0,14.0,9.0,10.0,6.0


## 3. 查詢解析工具（地點定位 + 時間轉換）

- `resolve_destination`：把地名/座標轉成 `dict(name, x, y, lat, lng)`。
  優先序為「直接座標 → 內建地標 gazetteer → Google Geocoding（若金鑰可用）」。
- `time_to_slot`：把 `HH:MM`（或 `1400` 這種無冒號格式）轉成 0–47 的時間槽。

> 內建地標的經緯度直接採用錨點實際座標；停車場座標則一律以 `mapper` 由格網推算，
> 兩者在同一投影空間下比較距離，內部一致。

In [4]:
# 內建地標：name -> (x, y, lat, lng)，離線即可定位
LANDMARKS = {
    "札幌車站": (24, 151, 43.0691833, 141.3514707),
    "札幌駅":   (24, 151, 43.0691833, 141.3514707),
    "sapporo station": (24, 151, 43.0691833, 141.3514707),
    "北海道大學": (24, 148, 43.0794037, 141.3422559),
    "北海道大学": (24, 148, 43.0794037, 141.3422559),
    "北大":      (24, 148, 43.0794037, 141.3422559),
    "hokkaido university": (24, 148, 43.0794037, 141.3422559),
}

def resolve_destination(name=None, latlng=None):
    """回傳目的地 dict(name, x, y, lat, lng)。"""
    if latlng is not None:
        lat, lng = float(latlng[0]), float(latlng[1])
        x, y = mapper.latlng_to_grid(lat, lng)
        return {"name": name or f"({lat:.4f},{lng:.4f})", "x": x, "y": y, "lat": lat, "lng": lng}

    key = (name or "").strip().lower()
    for k, v in LANDMARKS.items():
        if k.lower() == key or k.lower() in key or key in k.lower():
            x, y, lat, lng = v
            return {"name": name, "x": x, "y": y, "lat": lat, "lng": lng}

    # 找不到地標 -> 嘗試 Google Geocoding（需金鑰且已啟用 Geocoding API）
    gkey = os.getenv("GOOGLE_MAPS_API_KEY")
    if gkey:
        from src.confidence_engine import ConfidenceEngine
        info = ConfidenceEngine(df_feat, mapper, google_api_key=gkey).resolve_place_to_grid(name, CITY_BOUNDS)
        if info:
            return {"name": name, "x": info["x"], "y": info["y"], "lat": info["lat"], "lng": info["lng"]}
    raise ValueError(f"無法定位「{name}」：請改用內建地標、給定 latlng，或啟用 Google Geocoding API")

def time_to_slot(hhmm):
    """'14:00' 或 '1400' 或 '900' -> 0..47 時間槽"""
    s = str(hhmm).strip()
    if ":" in s:
        h, m = s.split(":")
    elif s.isdigit():
        s = s.zfill(4)
        h, m = s[:2], s[2:]
    else:
        return 24
    return (int(h) * 60 + int(m)) // 30

WEEKDAY_NAMES = ["週日", "週一", "週二", "週三", "週四", "週五", "週六"]
print("工具就緒：resolve_destination / time_to_slot")

工具就緒：resolve_destination / time_to_slot


## 4. 核心流程函式：`find_parking_around`

輸入「目的地 + 星期 + 時段 + 半徑」，依序執行：

1. **人流預測**：取目的地周邊格網的特徵，餵入 ConvLSTM。
2. **停車機率**：`create_poi_parking` 把人流轉車流需求，算出每個停車場各 (d, t) 的 `p_avail`，再依 weekday 聚合。
3. **半徑篩選**：換算停車場經緯度、計算與目的地球面距離，只留 ≤ 半徑者。
4. 整理輸出表（經緯度、容量、可停機率、預估空位、距離）。

In [12]:
def _neighbor_grids(x, y, r):
    return [(x + dx, y + dy) for dx in range(-r, r + 1) for dy in range(-r, r + 1)]

def find_parking_around(dest, weekday, target_t, radius_m, verbose=True):
    # 1) 半徑換算成格數，做人流預測 -----------------------------------------
    cell_m = cell_size_m_at(mapper, dest["x"], dest["y"])
    radius_cells = int(np.clip(np.ceil(radius_m / max(cell_m, 1e-6)), 1, 8))
    grid_df = pd.DataFrame(sorted(set(_neighbor_grids(dest["x"], dest["y"], radius_cells))),
                           columns=["x", "y"])
    t_lo, t_hi = target_t, min(target_t + T_BUFFER, 47)
    df_feat_small = (df_feat[(df_feat["t"] >= t_lo) & (df_feat["t"] <= t_hi)]
                     .merge(grid_df, on=["x", "y"], how="inner"))
    assert not df_feat_small.empty, "該時間/範圍內沒有可用特徵資料"
    df_pred = make_df_pred_convlstm_embed(df_feat_small, model, cfg)
    if verbose:
        print(f"[1] 一格約 {cell_m:.0f} m -> ±{radius_cells} 格；預測 {df_pred.shape[0]} 列 / "
              f"{df_pred[['x','y']].drop_duplicates().shape[0]} 格網")

    # 2) 停車機率 + weekday 聚合 --------------------------------------------
    south, west, north, east = CITY_BOUNDS
    _, _, df_prob = create_poi_parking(df_pred, south, west, north, east,
                                       mapper, START_DATE, refetch=False)
    df_prob_w = aggregate_df_prob_same_weekday(df_prob, agg="median")
    if verbose:
        print(f"[2] 涵蓋停車場 {df_prob_w['park_id'].nunique()} 個")

    # 3) 取查詢時段快照 ------------------------------------------------------
    snap = df_prob_w[(df_prob_w["w"] == weekday) & (df_prob_w["t"] == target_t)].copy()
    if snap.empty:  # 後備：該 weekday 無資料 -> 同時段跨 weekday 中位數
        agg_cols = [c for c in ["p_avail", "demand", "pressure"] if c in df_prob_w.columns]
        snap = (df_prob_w[df_prob_w["t"] == target_t]
                .groupby("park_id", as_index=False)[agg_cols].median()
                .merge(df_prob_w.drop_duplicates("park_id")[["park_id", "park_x", "park_y", "capacity"]],
                       on="park_id", how="left"))
        if verbose:
            print("   ⚠ 指定 weekday 無資料，改用同時段跨 weekday 中位數")
    snap = snap.drop_duplicates("park_id").reset_index(drop=True)

    # 格網 -> 經緯度，計算與目的地距離
    ll = mapper.transform(snap.rename(columns={"park_x": "x", "park_y": "y"})[["x", "y"]])
    snap["lat"], snap["lng"] = ll["lat"].to_numpy(), ll["lng"].to_numpy()
    snap["dist_m"] = haversine_m_vec(snap["lat"].to_numpy(), snap["lng"].to_numpy(),
                                     dest["lat"], dest["lng"])

    # 4) 半徑內 + 預估空位 ---------------------------------------------------
    res = snap[snap["dist_m"] <= radius_m].copy()
    cap = res["capacity"].astype(float)
    if "demand" in res.columns:
        res["free_spots"] = np.clip(np.round(cap - np.minimum(res["demand"].astype(float), cap)), 0, None).astype(int)
    else:
        res["free_spots"] = np.round(cap * res["p_avail"].astype(float)).astype(int)
    res["walk_min"] = res["dist_m"] / 1000.0 / 4.8 * 60
    res = res.sort_values("p_avail", ascending=False).reset_index(drop=True)
    if verbose:
        print(f"[3] 半徑 {radius_m:.0f} m 內找到 {len(res)} 個停車場")
    return res

def build_map(result, dest, radius_m, save_path="outputs/destination_radius_parking.html"):
    def prob_color(p):
        return "green" if p >= 0.6 else ("orange" if p >= 0.3 else "red")

    fmap = folium.Map(location=[dest["lat"], dest["lng"]], zoom_start=14, tiles="cartodbpositron")
    folium.Marker([dest["lat"], dest["lng"]],
                  popup=f"<b>目的地：{dest['name']}</b><br>{dest['lat']:.5f}, {dest['lng']:.5f}",
                  icon=folium.Icon(color="red", icon="flag")).add_to(fmap)
    folium.Circle([dest["lat"], dest["lng"]], radius=radius_m, color="#2C3E50",
                  weight=2, fill=True, fill_opacity=0.06,
                  popup=f"搜尋半徑 {radius_m:.0f} m").add_to(fmap)

    for _, r in result.iterrows():
        folium.Marker(
            [float(r["lat"]), float(r["lng"])],
            icon=folium.Icon(color=prob_color(float(r["p_avail"])), icon="car", prefix="fa"),
            popup=folium.Popup(
                f"<b>{r['park_id']}</b><br>"
                f"經緯度: {r['lat']:.5f}, {r['lng']:.5f}<br>"
                f"容量: {int(r['capacity'])} 位<br>"
                f"可停機率: {float(r['p_avail']):.1%}<br>"
                f"預估空位: {int(r['free_spots'])} 位<br>"
                f"距目的地: {r['dist_m']:.0f} m（步行約 {r['walk_min']:.0f} 分）",
                max_width=260),
            tooltip=f"{r['park_id']}  可停 {float(r['p_avail']):.0%}").add_to(fmap)

    pts = [(dest["lat"], dest["lng"])] + list(zip(result["lat"], result["lng"]))
    if len(pts) > 1:
        lats, lngs = zip(*pts)
        fmap.fit_bounds([[min(lats), min(lngs)], [max(lats), max(lngs)]])
    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        fmap.save(save_path)
        print(f"地圖已存檔：{save_path}")
    return fmap

print("流程函式就緒：find_parking_around / build_map")

流程函式就緒：find_parking_around / build_map


## 5. 執行查詢（結構化輸入，可離線）

改這 5 個變數即可：目的地、星期（`0=週日…6=週六`）、時間、半徑（= 示意圖中的 **X**）。

In [16]:
# ====== 查詢輸入 ======
DEST_NAME   = "丘珠空港"   # 或 "札幌車站"
DEST_LATLNG = None            # 例如 (43.0794, 141.3423)；設了就優先於 DEST_NAME
WEEKDAY     = 6               # 6 = 週六
HHMM        = "14:00"        # 下午 2 點
RADIUS_M    = 1000.0          # 搜尋半徑（公尺）= 示意圖中的 X
# ======================

dest = resolve_destination(DEST_NAME, DEST_LATLNG)
target_t = time_to_slot(HHMM)
print(f"目的地：{dest['name']} -> grid({dest['x']},{dest['y']}) ({dest['lat']:.5f}, {dest['lng']:.5f})")
print(f"時間：{WEEKDAY_NAMES[WEEKDAY]} {HHMM}（slot t={target_t}）｜半徑 {RADIUS_M:.0f} m\n")

result = find_parking_around(dest, WEEKDAY, target_t, RADIUS_M)

show_cols = ["park_id", "lat", "lng", "capacity", "p_avail", "free_spots", "dist_m", "walk_min"]
result[show_cols].round({"lat": 5, "lng": 5, "p_avail": 3, "dist_m": 0, "walk_min": 1}).head(20)

Google Geocode: 丘珠空港 -> 43.1158533,141.3801858
✓ 丘珠空港 -> (43.11585,141.38019) -> grid(34,157)
目的地：丘珠空港 -> grid(34,157) (43.11585, 141.38019)
時間：週六 14:00（slot t=28）｜半徑 1000 m



c:\Users\USER\Desktop\2026_japan\venv312\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


[1] 一格約 482 m -> ±3 格；預測 10826 列 / 42 格網
[2] 涵蓋停車場 42 個
[3] 半徑 1000 m 內找到 7 個停車場


,park_id,lat,lng,capacity,p_avail,free_spots,dist_m,walk_min
0,way/155426508,43.11306,141.36974,141,0.469,141,903.0,11.3
1,way/1143910411,43.11308,141.36897,72,0.466,72,961.0,12.0
2,way/155518215,43.11713,141.37369,129,0.456,128,546.0,6.8
3,way/1135839336,43.10758,141.37559,65,0.447,64,993.0,12.4
4,way/1135839154,43.10802,141.37555,18,0.420,18,949.0,11.9
5,way/155371024,43.11006,141.38152,29,0.391,28,653.0,8.2
6,way/1089577500,43.12230,141.38330,37,0.370,36,760.0,9.5


## 6. 標註在地圖上

目的地以紅旗標示並畫出半徑 X 的搜尋圈；停車場依可停機率上色
（🟢 高 ≥0.6 / 🟠 中 ≥0.3 / 🔴 低）。點選 marker 可看經緯度、容量、可停機率與預估空位。

In [17]:
fmap = build_map(result, dest, RADIUS_M)
fmap

地圖已存檔：outputs/destination_radius_parking.html


## 附錄：自然語言查詢（需 `OPENAI_API_KEY`）

用一句中文自動解析目的地與時間，再跑同一套 `find_parking_around` 流程。
目的地一律先用內建地標定位（即使 Google Geocoding 未啟用也能運作）。

In [8]:
NL_QUERY = "星期六下午2點到北海道大學，附近1400公尺有沒有車位"

if os.getenv("OPENAI_API_KEY"):
    from openai import OpenAI
    from src.confidence_engine import ConfidenceEngine

    eng = ConfidenceEngine(df_feat, mapper, openai_client=OpenAI(),
                           google_api_key=os.getenv("GOOGLE_MAPS_API_KEY"))
    q = eng.parse_route_query_llm(NL_QUERY)
    print("LLM 解析：", q)

    dest_nl = resolve_destination(q.dest_place)            # 先走 gazetteer，失敗才打 Google
    t_nl    = time_to_slot(q.hhmm) if q.hhmm else 28
    wd_nl   = int(q.weekday) if q.weekday is not None else 6
    rad_nl  = float(q.radius_m) if q.radius_m else 1400.0
    print(f"-> {dest_nl['name']} grid({dest_nl['x']},{dest_nl['y']}), "
          f"{WEEKDAY_NAMES[wd_nl]}, slot {t_nl}, 半徑 {rad_nl:.0f} m\n")

    result_nl = find_parking_around(dest_nl, wd_nl, t_nl, rad_nl)
    display(result_nl[show_cols].round({"lat": 5, "lng": 5, "p_avail": 3,
                                        "dist_m": 0, "walk_min": 1}).head(20))
    build_map(result_nl, dest_nl, rad_nl,
              save_path="outputs/destination_radius_parking_nlq.html")
else:
    print("未偵測到 OPENAI_API_KEY，略過自然語言查詢（第 5、6 節的結構化輸入已足以完成 Demo）。")

LLM 解析： RouteQuery(city='札幌市', start_place='', dest_place='北海道大學', weekday=6, hhmm='1400', radius_m=1400)
-> 北海道大學 grid(24,148), 週六, slot 28, 半徑 1400 m



c:\Users\USER\Desktop\2026_japan\venv312\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


[1] 一格約 482 m -> ±3 格；預測 14257 列 / 48 格網
[2] 涵蓋停車場 498 個
[3] 半徑 1400 m 內找到 92 個停車場


,park_id,lat,lng,capacity,p_avail,free_spots,dist_m,walk_min
0,way/914785212,43.07852,141.35257,103,0.473,103,843.0,10.5
1,way/1101891330,43.06968,141.35055,230,0.473,230,1274.0,15.9
2,way/508035100,43.07002,141.35198,61,0.470,61,1308.0,16.4
3,way/1110297680,43.07455,141.35314,29,0.468,29,1036.0,12.9
4,way/162680904,43.08196,141.34547,65,0.468,65,386.0,4.8
5,way/573804182,43.08138,141.35036,20,0.467,20,694.0,8.7
6,way/508300771,43.07331,141.35291,25,0.467,25,1099.0,13.7
7,way/914785239,43.07528,141.35140,39,0.467,39,872.0,10.9
8,node/3023890933,43.06930,141.35250,24,0.466,24,1398.0,17.5
9,way/914785246,43.07920,141.34546,96,0.466,96,261.0,3.3


地圖已存檔：outputs/destination_radius_parking_nlq.html


In [9]:
# 顯示自然語言查詢的地圖（若上一格有產生）
try:
    build_map(result_nl, dest_nl, rad_nl, save_path=None)
except NameError:
    print("尚未執行自然語言查詢，無地圖可顯示。")